# TUGAS 3: Web Crawler & Information Extraction Putusan Mahkamah Agung RI
### Mata Kuliah: Temu Kembali Informasi (TKI) — Semester Ganjil 2026/2027
**Program Studi Teknik Informatika, Fakultas Teknik, Universitas Muhammadiyah Malang**

* **Mahasiswa**: Muhammad Aulia Rahman
* **NIM**: 202310370311005
* **Dosen Pengampu**: Galih W. W

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hiroyuuki01/Project-TKI/blob/main/Scrapping.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-Project--TKI-blue.svg)](https://github.com/Hiroyuuki01/Project-TKI)

---

## 📌 Catatan Lingkungan Eksekusi Web Crawler (PENTING)

Program *Web Crawler* ini dirancang dan **dijalankan secara riil menggunakan VS Code (Komputer Lokal / Desktop)**.

### Mengapa Dijalankan di VS Code (Lokal)?
1. **Proteksi Cloudflare Turnstile**:
   Website Direktori Putusan Mahkamah Agung RI (`https://putusan3.mahkamahagung.go.id`) dilindungi oleh sistem keamanan Cloudflare Anti-Bot. Permintaan HTTP tanpa peramban riil (*Requests biasa, cURL, atau IP cloud*) langsung diblokir dengan kode status **HTTP 403 Forbidden**.
2. **Pemanfaatan `DrissionPage`**:
   Program ini menggunakan pustaka `DrissionPage` yang mengendalikan peramban Google Chrome fisik di laptop/desktop sehingga proses perayapan dan pengunduhan PDF berjalan secara riil, legal, dan berhasil melewati verifikasi Cloudflare.
3. **Kepatuhan Kesusilaan `robots.txt`**:
   Crawler memeriksa berkas kebijakan kesusilaan `https://putusan3.mahkamahagung.go.id/robots.txt` dan menerapkan jeda *Crawl-delay* 3 detik antar-halaman.

> **Catatan untuk Google Colaboratory**:
> Lingkungan Google Colaboratory berbasis server cloud Linux *headless* (tanpa tampilan monitor fisik dan tanpa instalasi aplikasi Chrome desktop). Jika notebook ini dibuka di Google Colab, program memuat seluruh kode asli yang kami gunakan untuk akuisisi data. Namun, untuk menjalankan ulang proses perayapan langsung (*live crawling*) ke server Mahkamah Agung, program wajib dieksekusi di komputer lokal (VS Code) dengan browser Chrome aktif.

---
## Cell 1: Instalasi & Pemuatan Library

In [ ]:
import sys
import os
import subprocess

# Menginstal library ke kernel Python
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "DrissionPage", "pdfplumber", "beautifulsoup4", "pandas", "openpyxl"
])

import re
import time
import glob
import pandas as pd
from bs4 import BeautifulSoup
import pdfplumber
from DrissionPage import ChromiumPage, ChromiumOptions

print("[OK] Cell 1 Selesai: Semua library berhasil terpasang dan siap digunakan!")

---
## Cell 2: Konfigurasi Direktori Sasaran & Folder Output

In [ ]:
# Target direktori perkara Narkotika PN Banjarmasin
TARGET_URL_DIREKTORI = "https://putusan3.mahkamahagung.go.id/direktori/index/pengadilan/pn-banjarmasin/kategori/narkotika-dan-psikotropika-1.html"

NAMA_LEMBAGA_DEFAULT = "PN Banjarmasin"
TARGET_JUMLAH_PDF = 50
MAKSIMAL_HALAMAN_DIREKTORI = 15

# Output sesuai kriteria penugasan LMS:
PDF_DIR = "PDF_Putusan"
TXT_DIR = "TXT_Putusan"
CSV_OUTPUT = "Hasil_Ekstraksi_Tugas3.csv"

os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(TXT_DIR, exist_ok=True)

print(f"Target Direktori : {TARGET_URL_DIREKTORI}")
print(f"Folder PDF       : {PDF_DIR}/")
print(f"Folder TXT       : {TXT_DIR}/")
print(f"File CSV Output  : {CSV_OUTPUT}")
print("[OK] Cell 2 Selesai: Folder output siap digunakan!")

---
## Cell 3: Fungsi Pembersihan Noise Watermark & Parser Regex
Menyaring teks watermark vertikal font *Helvetica* Mahkamah Agung dan mengekstrak seluruh entitas yang disyaratkan.

In [ ]:
def extract_text_from_pdf(pdf_path, txt_output_dir=TXT_DIR):
    """
    Membaca teks dari file PDF dengan menyaring watermark ber-font Helvetica.
    Menyimpan hasil teks bersih ke file .txt di folder TXT_Putusan.
    """
    pages_text = []
    with pdfplumber.open(pdf_path) as pdf:
        num_pages = len(pdf.pages)
        for page in pdf.pages:
            try:
                # Saring karakter watermark yang ber-font Helvetica
                clean_page = page.filter(
                    lambda obj: not (obj.get("object_type") == "char" and "Helvetica" in obj.get("fontname", ""))
                )
                t = clean_page.extract_text()
                if t:
                    pages_text.append(t)
            except Exception:
                t = page.extract_text()
                if t:
                    pages_text.append(t)
                    
    full_text = "\n".join(pages_text)
    
    if txt_output_dir:
        os.makedirs(txt_output_dir, exist_ok=True)
        base_name = os.path.splitext(os.path.basename(pdf_path))[0]
        txt_path = os.path.join(txt_output_dir, f"{base_name}.txt")
        with open(txt_path, "w", encoding="utf-8") as f_txt:
            f_txt.write(full_text)
            
    return full_text, num_pages


def parse_putusan_narkotika(text, default_lembaga=NAMA_LEMBAGA_DEFAULT):
    """
    Mengekstrak entitas informasi putusan ke format tabel Tugas 3:
    - Kolom Wajib: No Putusan, Lembaga Peradilan, Barang Bukti, Amar Putusan
    - Kolom Nilai Tambah (Poin Plus): Identitas Terdakwa, Tuntutan JPU
    """
    flat_text = re.sub(r"\s+", " ", text)

    # 1. No Putusan
    no_putusan = "-"
    m_no = re.search(r"(?:PUTUSAN\s*)?(?:Nomor|No\.?)\s*[:\.]?\s*([0-9]+\s*/\s*Pid\.Sus\s*/\s*20[0-9]{2}\s*/\s*PN\s*[A-Za-z0-9\.]+)", text, re.I)
    if not m_no:
        m_no = re.search(r"([0-9]+\s*/\s*Pid\.Sus\s*/\s*20[0-9]{2}\s*/\s*PN\s*[A-Za-z0-9\.]+)", text, re.I)
    if m_no:
        no_putusan = re.sub(r"\s+", "", m_no.group(1))

    # 2. Lembaga Peradilan
    lembaga = default_lembaga
    m_pn = re.search(r"Pengadilan\s+Negeri\s+([A-Za-z\s]+?)(?:\s+yang|\s+Kelas|\s+pada|\n)", text, re.I)
    if m_pn:
        lembaga = f"PN {m_pn.group(1).strip()}"

    # 3. Identitas Terdakwa (Poin Nilai Tambah)
    identitas_terdakwa = "-"
    m_nama = re.search(r"Nama\s+Lengkap\s*:\s*([^\n:;]+?)(?:\s+Tempat\s+Lahir|\s+Umur|\s+Jenis\s+Kelamin|\n)", text, re.I)
    if m_nama:
        identitas_terdakwa = m_nama.group(1).strip()
    else:
        m_terdakwa_alt = re.search(r"Menyatakan\s+(?:Para\s+)?Terdakwa\s+([A-Z\s\.,']+?)\s+(?:tersebut\s+di\s*atas|telah\s+terbukti|terbukti)", text)
        if m_terdakwa_alt:
            identitas_terdakwa = m_terdakwa_alt.group(1).strip()

    # 4. Tuntutan JPU (Poin Nilai Tambah)
    tuntutan_jpu = "-"
    m_tuntutan = re.search(
        r"(?:tuntutan\s+pidana\s+Penuntut\s+Umum|menuntut\s+supaya\s+Majelis\s+Hakim|pada\s+pokoknya\s+menuntut)[\s\S]{1,250}?(?:sebagai\s+berikut|memutuskan)\s*:\s*(1\.[^;]+?(?:Menimbang|Setelah\s+mendengar))",
        text, re.I
    )
    if m_tuntutan:
        tuntutan_jpu = re.sub(r"\s+", " ", m_tuntutan.group(1)).strip()

    # 5. Amar Putusan (Wajib)
    amar_putusan = "-"
    m_amar = re.search(
        r"(?:^|\n)\s*M\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I\s*:?\s*\n([\s\S]+?)(?:\n\s*Demikian(?:lah)?\s+diputus|\n\s*Hakim\s+Anggota|\n\s*Panitera\s+Pengganti)",
        text, re.I
    )
    if not m_amar:
        m_amar = re.search(r"\bMENGADILI\s*:?\s*([\s\S]+?)(?:Demikian(?:lah)?\s+diputus|Hakim\s+Anggota|Panitera\s+Pengganti)", flat_text, re.I)
    if m_amar:
        raw_amar = m_amar.group(1).strip()
        amar_putusan = re.sub(r"\n{2,}", "\n", raw_amar)
        amar_putusan = re.sub(r"[ \t]+", " ", amar_putusan)

    # 6. Barang Bukti (Wajib)
    barang_bukti = "-"
    search_bb = amar_putusan if amar_putusan != "-" else text
    m_bb = re.search(
        r"Menetapkan\s+barang\s+bukti\s+berupa\s*:\s*([\s\S]+?)(?:Dirampas|Dikembalikan|Tetap\s+terlampir|Membebankan\s+kepada|Menetapkan|\d+\.\s*Membebankan)",
        search_bb, re.I
    )
    if m_bb:
        bb_clean = re.sub(r"\s+", " ", m_bb.group(1)).strip(" -;:\n")
        barang_bukti = bb_clean + ";"

    return {
        "No Putusan": no_putusan,
        "Lembaga Peradilan": lembaga,
        "Barang Bukti": barang_bukti,
        "Amar Putusan": amar_putusan,
        "Identitas Terdakwa": identitas_terdakwa,
        "Tuntutan JPU": tuntutan_jpu
    }

print("[OK] Cell 3 Selesai: Fungsi pembersihan teks dan parser Regex siap!")

---
## Cell 4: Modul Web Crawler (DrissionPage — Browser Chrome Lokal Anti-403)
Menelusuri direktori Mahkamah Agung RI, mematuhi jeda `robots.txt`, dan mengunduh berkas PDF putusan secara nyata.

In [ ]:
def wait_if_cloudflare(page):
    """Menunggu apabila jendela Chrome menampilkan verifikasi Cloudflare."""
    for _ in range(15):
        title = (page.title or "").lower()
        if "just a moment" in title or "tunggu sebentar" in title or "cloudflare" in title:
            print("   [CLOUDFLARE] Silakan klik centang verifikasi Cloudflare di jendela Chrome...")
            time.sleep(3)
        else:
            break


def crawl_and_download_pdfs():
    """Merayapi direktori MA dan mengunduh file PDF asli via browser Chrome fisik."""
    print("=" * 75)
    print("MULAI WEB CRAWLING DIREKTORI PUTUSAN MAHKAMAH AGUNG (TUGAS 3)")
    print("=" * 75)

    abs_pdf_dir = os.path.abspath(PDF_DIR)
    co = ChromiumOptions()
    co.set_paths(download_path=abs_pdf_dir)
    co.set_pref("plugins.always_open_pdf_externally", True)
    co.set_pref("download.default_directory", abs_pdf_dir)

    page = ChromiumPage(co)
    page.set.download_path(abs_pdf_dir)

    # 1. Cek robots.txt sesuai SUBCPMK-3
    robots_url = "https://putusan3.mahkamahagung.go.id/robots.txt"
    print(f"[1] Memeriksa robots.txt: {robots_url}")
    page.get(robots_url)
    wait_if_cloudflare(page)
    time.sleep(2)

    # 2. Ambil daftar link putusan dari halaman direktori
    collected_detail_urls = []
    for hal in range(1, MAKSIMAL_HALAMAN_DIREKTORI + 1):
        if len(collected_detail_urls) >= TARGET_JUMLAH_PDF * 2:
            break

        url_hal = TARGET_URL_DIREKTORI if hal == 1 else TARGET_URL_DIREKTORI.replace(".html", f"/page/{hal}.html")
        print(f"\n[2] Membuka Halaman Direktori ke-{hal}: {url_hal}")
        page.get(url_hal)
        wait_if_cloudflare(page)
        time.sleep(3) # Jeda Crawl-Delay

        soup = BeautifulSoup(page.html, "html.parser")
        links_on_page = 0
        for a_tag in soup.select("a[href*='/direktori/putusan/']"):
            href = a_tag.get("href")
            if href and href not in collected_detail_urls:
                collected_detail_urls.append(href)
                links_on_page += 1
        print(f"    -> Ditemukan {links_on_page} link putusan pada halaman ke-{hal}.")

    # 3. Buka setiap halaman putusan dan klik unduh PDF
    print(f"\n[3] Mengunduh {TARGET_JUMLAH_PDF} PDF Putusan ke folder '{PDF_DIR}/'...")
    downloaded_count = 0

    for idx, detail_url in enumerate(collected_detail_urls, start=1):
        if downloaded_count >= TARGET_JUMLAH_PDF:
            break

        print(f"\n   [{idx}] Membuka halaman putusan: {detail_url}")
        page.get(detail_url)
        wait_if_cloudflare(page)
        time.sleep(3)

        pdf_ele = page.ele("css:a[href*='/direktori/download_file/'][href*='/pdf/']", timeout=3)
        if not pdf_ele:
            pdf_ele = page.ele("css:a[href*='/direktori/download_file/']", index=2, timeout=2)

        if not pdf_ele:
            print("       -> [SKIP] Halaman putusan tidak memiliki lampiran PDF.")
            continue

        pdf_url = pdf_ele.attr("href")
        if "/zip/" in str(pdf_url):
            print("       -> [SKIP] Link mengarah ke ZIP, melewati...")
            continue

        nama_file_pdf = f"Putusan_{downloaded_count + 1:02d}.pdf"
        print(f"       -> Mengunduh PDF via Browser Chrome: {pdf_url}")

        try:
            mission = pdf_ele.click.to_download(
                save_path=abs_pdf_dir,
                rename=nama_file_pdf,
                timeout=45
            )
            page.wait.downloads_done()
            
            saved_path = os.path.join(abs_pdf_dir, nama_file_pdf)
            if mission and os.path.exists(saved_path) and os.path.getsize(saved_path) > 1000:
                downloaded_count += 1
                print(f"       -> [BERHASIL ({downloaded_count}/{TARGET_JUMLAH_PDF})] Tersimpan: {saved_path}")
            else:
                print("       -> [GAGAL] Unduhan tidak selesai atau file kosong.")
        except Exception as err:
            print(f"       -> [ERROR] {err}")

    page.quit()
    print(f"\n[SELESAI CRAWLING] Total PDF berhasil diunduh: {downloaded_count} file di folder '{PDF_DIR}/'.")

print("[OK] Cell 4 Selesai: Fungsi Web Crawler Anti-403 siap dijalankan!")

---
## Cell 5: Eksekusi Web Crawling & Ekstraksi ke CSV

In [ ]:
# 1. Jalankan Crawler (Jalankan cell ini di VS Code Lokal untuk meng-crawl langsung)
try:
    crawl_and_download_pdfs()
except Exception as e:
    print(f"[CATATAN EKSEKUSI CRAWLER]: {e}")
    print(f"-> Jika dijalankan di cloud/Colab tanpa GUI Chrome fisik, crawling dilewati.")
    print(f"-> Melanjutkan proses ekstraksi pada dokumen di '{PDF_DIR}/'...")

# 2. Ekstraksi Dokumen PDF ke TXT Bersih dan Parsing ke CSV
pdf_files = sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")))
records = []

if pdf_files:
    print(f"\nMemulai ekstraksi pada {len(pdf_files)} dokumen putusan...\n")
    for idx, pdf_path in enumerate(pdf_files, start=1):
        fname = os.path.basename(pdf_path)
        try:
            text, num_pages = extract_text_from_pdf(pdf_path, txt_output_dir=TXT_DIR)
            extracted = parse_putusan_narkotika(text, default_lembaga=NAMA_LEMBAGA_DEFAULT)
            row = {"No": idx}
            row.update(extracted)
            records.append(row)
            
            if idx <= 5 or idx % 10 == 0 or idx == len(pdf_files):
                print(f"[{idx:02d}/{len(pdf_files)}] {fname} -> {extracted['No Putusan']} | Amar: {'V' if extracted['Amar Putusan'] != '-' else 'X'} | BB: {'V' if extracted['Barang Bukti'] != '-' else 'X'}")
        except Exception as e:
            print(f"[{idx:02d}/{len(pdf_files)}] Gagal memproses {fname}: {e}")

    df_tugas3 = pd.DataFrame(records)
    if not df_tugas3.empty:
        df_tugas3.to_csv(CSV_OUTPUT, index=False, encoding="utf-8-sig")
        print("\n" + "=" * 75)
        print("PENGELOLAAN & EKSTRAKSI TUGAS 3 BERHASIL!")
        print(f"1. Folder PDF Putusan : {os.path.abspath(PDF_DIR)}")
        print(f"2. Folder TXT Putusan : {os.path.abspath(TXT_DIR)}")
        print(f"3. File CSV Ekstraksi : {os.path.abspath(CSV_OUTPUT)}")
        print(f"Total Baris Data      : {len(df_tugas3)} putusan")
        print("=" * 75)

    df_tugas3.head(5)
else:
    print(f"[INFO] Folder '{PDF_DIR}' belum berisi dokumen PDF.")